# 📂 Kaggle CSV Merger: Robust Multi-File Concatenation

This notebook provides a robust, beginner-friendly solution for merging multiple CSV files from a Kaggle input directory into a single, unified dataset. It handles inconsistent column structures, encoding issues, and empty files automatically.

### Key Features:
- **Automatic Discovery**: Finds all `.csv` files in a specified directory.
- **Column Alignment**: Performs an outer join to include all unique columns from all files.
- **Error Handling**: Gracefully handles encoding errors and empty files.
- **Kaggle Ready**: Configured for `/kaggle/input/` and `/kaggle/working/` paths.

## 📦 Step 1: Import Libraries
We use `pandas` for data manipulation, `os` for path handling, and `glob` for file discovery.

In [1]:
import pandas as pd
import os
import glob
import warnings

warnings.filterwarnings('ignore')
print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


## ⚙️ Step 2: Configuration
Define the input directory and the final output filename. 

> **Note**: Update `INPUT_DIR` to point to your specific dataset folder inside `/kaggle/input/`.

> **Update**: This notebook can also be used on your local machine. Just set `INPUT_DIR` and `OUTPUT_FILE` to your local folder paths (e.g., `INPUT_DIR = './my_csv_folder/'`, `OUTPUT_FILE = './final_merged.csv'`). All other steps remain the same.

In [2]:
# Path to the folder containing your CSV files
INPUT_DIR = '/home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Topic_Wise_w3/'  # Update this to your specific dataset path
OUTPUT_FILE = '/home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/DATA_W3.csv'

REMOVE_DUPLICATES = True  # Set to False if you want to keep duplicates

print(f"📂 Input Directory: {INPUT_DIR}")
print(f"💾 Output File: {OUTPUT_FILE}")

📂 Input Directory: /home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Topic_Wise_w3/
💾 Output File: /home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/DATA_W3.csv


## 🔍 Step 3: File Discovery
We use `glob` to find all CSV files recursively if needed.

In [3]:
# Find all CSV files in the input directory
csv_files = glob.glob(os.path.join(INPUT_DIR, '**/*.csv'), recursive=True)

print(f"🔢 Found {len(csv_files)} CSV files.")
for f in csv_files[:5]:  # Show first 5 files
    print(f"   - {os.path.basename(f)}")
if len(csv_files) > 5:
    print("   ... and more.")

🔢 Found 5 CSV files.
   - Climate.csv
   - Health.csv
   - Technology.csv
   - Economics.csv
   - War.csv


In [4]:
# Show columns and their data types in the first CSV file
if csv_files:
    try:
        first_df = pd.read_csv(csv_files[0])
        print(f"Columns in the first file ({os.path.basename(csv_files[0])}):")
        for col, dtype in zip(first_df.columns, first_df.dtypes):
            print(f"   - {col}: {dtype}")
    except Exception as e:
        print(f"❌ Error reading first file columns: {e}")
else:
    print("No CSV files found.")

Columns in the first file (Climate.csv):
   - date: str
   - sentence_id: str
   - main_sentence: str
   - w3_embedding: str
   - War: float64
   - Health: float64
   - Technology: float64
   - Climate: float64
   - Economics: float64


## 🔄 Step 4: Iterative Merging
We loop through each file, read it into a DataFrame, and store it in a list for final concatenation. This approach is memory-efficient and robust.

In [ ]:
import csv

from collections import OrderedDict

import pandas as pd

import os

import glob

import warnings

warnings.filterwarnings('ignore')



# Initialize output file

first_file = True

processed_count = 0

error_count = 0

columns_union = OrderedDict()



for idx, file_path in enumerate(csv_files):

    try:

        # Try reading with UTF-8 encoding first

        try:

            df = pd.read_csv(file_path)

        except UnicodeDecodeError:

            df = pd.read_csv(file_path, encoding='latin1')

        if df.empty:

            print(f"⚠️ Skipping empty file: {os.path.basename(file_path)}")

            continue

        # Update columns union

        for col in df.columns:

            columns_union[col] = None

        if first_file:

            df.to_csv(OUTPUT_FILE, index=False, mode='w')

            first_file = False

            print(f"✨ Written first file: {os.path.basename(file_path)}")

        else:

            # Align columns to union

            for col in columns_union:

                if col not in df.columns:

                    df[col] = pd.NA

            df = df[list(columns_union.keys())]

            df.to_csv(OUTPUT_FILE, index=False, mode='a', header=False)

            print(f"➕ Appended: {os.path.basename(file_path)}")

        processed_count += 1

    except Exception as e:

        print(f"❌ Error processing {os.path.basename(file_path)}: {e}")

        error_count += 1



print(f"\n✅ Successfully processed {processed_count} files.")

if error_count > 0:

    print(f"⚠️ Encountered errors in {error_count} files.")

✨ Written first file: Climate.csv
➕ Appended: Health.csv
➕ Appended: Technology.csv
➕ Appended: Economics.csv


## 🔗 Step 5: Concatenation & Post-Processing
We use `pd.concat` with `sort=False` to merge the DataFrames. Missing columns in any file will be filled with `NaN` automatically.

In [ ]:
if all_dfs:
    # Concatenate all DataFrames
    # sort=False keeps the original column order from the first file
    final_df = pd.concat(all_dfs, axis=0, ignore_index=True, sort=False)
    
    # Remove duplicates if enabled
    if REMOVE_DUPLICATES:
        initial_rows = len(final_df)
        final_df.drop_duplicates(inplace=True)
        removed_rows = initial_rows - len(final_df)
        print(f"✨ Removed {removed_rows} duplicate rows.")
    
    print(f"📊 Final Dataset Shape: {final_df.shape}")
else:
    print("❌ No data to merge.")

## 💾 Step 6: Export to CSV
Save the final merged dataset to the Kaggle working directory.

In [ ]:
if 'final_df' in locals():
    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"🚀 Successfully saved merged file to: {OUTPUT_FILE}")

## 📝 Step 7: Final Summary
Review the final columns and a quick preview of the data.

In [ ]:
if 'final_df' in locals():
    print("\n📋 Final Columns:")
    print(list(final_df.columns))
    
    print("\n🔎 Column Data Types:")
    print(final_df.dtypes)
    
    print("\n👀 Data Preview (First 5 rows):")
    display(final_df.head())
